# Dark Mode Adaptation — Test Variations

Compare different approaches for making Rich's Jupyter HTML readable on dark backgrounds.

**Toggle your notebook between light/dark mode to test each cell.**

## Setup: sample Rich HTML

In [ ]:
from IPython.display import HTML, display

# This is what Rich produces for a typical progress bar line.
# Unstyled text is bare (no color), styled text gets inline color.
SAMPLE_RICH_HTML = (
    '◈ llm-pipeline '
    '<span style="color: #008000; text-decoration-color: #008000">━━━━━━━━━━</span> '
    '100/100 0:00:05 96✓ '
    '<span style="color: #800000; text-decoration-color: #800000; font-weight: bold">4✗</span>'
)

FONT = "font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace"

def show(label, pre_style, wrapper=''):
    """Display a labeled test case."""
    html = f'<div style="margin:12px 0">'
    html += f'<div style="{FONT}; font-size:0.8em; color:gray; margin-bottom:4px">{label}</div>'
    if wrapper:
        html += f'{wrapper}<pre style="{pre_style}">{SAMPLE_RICH_HTML}</pre></div>'
    else:
        html += f'<pre style="{pre_style}">{SAMPLE_RICH_HTML}</pre></div>'
    # Close wrapper if present
    if wrapper:
        html += '</div>'
    display(HTML(html))

---
## 1. Baseline — Rich's default (no fix)
This is what Rich produces out of the box. Text is black → invisible on dark.

In [ ]:
show(
    '1. Baseline (Rich default — no color on pre)',
    f'white-space:pre;overflow-x:auto;line-height:normal;{FONT}',
)

---
## 2. `color: inherit`
Inherits from parent. Often still black inside ipywidgets.Output.

In [ ]:
show(
    '2. color: inherit',
    f'white-space:pre;overflow-x:auto;line-height:normal;{FONT};color:inherit',
)

---
## 3. `color-scheme: light dark` + `light-dark()` (OS preference)
Uses CSS light-dark() which resolves based on OS color scheme preference.
Works when notebook theme matches OS theme. Fails when they differ.

In [ ]:
show(
    '3. color-scheme + light-dark() — follows OS preference',
    f'white-space:pre;overflow-x:auto;line-height:normal;{FONT};color-scheme:light dark;color:light-dark(#111827,#d1d5db)',
)

---
## 4. Wrapper div with theme detection JS + `light-dark()`
Same as our HTML widgets: a container div with `color-scheme: light dark`,
plus a tiny JS script that detects JupyterLab/VS Code/Marimo theme and
overrides the `colorScheme` property so `light-dark()` follows the notebook, not OS.

In [ ]:
THEME_JS = (
    "(function(){var c=document.currentScript.previousElementSibling;try{"
    "var b=document.body,r=document.documentElement,t=null;"
    "var jp=b.dataset.jpThemeLight;"
    "if(jp==='false')t='dark';else if(jp==='true')t='light';"
    "if(!t){var cn=b.className||'';"
    "if(cn.includes('jp-mod-dark'))t='dark';"
    "else if(cn.includes('jp-mod-light'))t='light';}"
    "var tk=b.getAttribute('data-vscode-theme-kind');"
    "if(tk)t=tk.includes('light')?'light':'dark';"
    "if(!t){var dt=b.dataset.theme||r.dataset.theme;"
    "var dm=b.dataset.mode||r.dataset.mode;"
    "if(dt==='dark'||dm==='dark')t='dark';"
    "else if(dt==='light'||dm==='light')t='light';}"
    "if(t)c.style.colorScheme=t;"
    "}catch(e){}})()"
)

pre_style = f'white-space:pre;overflow-x:auto;line-height:normal;{FONT};color:light-dark(#111827,#d1d5db)'
wrapper = f'<div style="color-scheme:light dark">'
# wrapper div + pre + close div + script
html = (
    f'<div style="margin:12px 0">'
    f'<div style="{FONT}; font-size:0.8em; color:gray; margin-bottom:4px">4. Wrapper div + theme JS + light-dark()</div>'
    f'<div style="color-scheme:light dark">'
    f'<pre style="{pre_style}">{SAMPLE_RICH_HTML}</pre>'
    f'</div>'
    f'<script>{THEME_JS}</script>'
    f'</div>'
)
display(HTML(html))

---
## 5. `@media (prefers-color-scheme: dark)` with `<style>` tag
Uses a CSS media query. Same limitation as #3 — follows OS, not notebook theme.
But some notebook renderers strip `<style>` tags entirely.

In [ ]:
html = (
    f'<div style="margin:12px 0">'
    f'<div style="{FONT}; font-size:0.8em; color:gray; margin-bottom:4px">5. @media prefers-color-scheme</div>'
    f'<style>.rich-adaptive-pre {{ color: #111827; }} '
    f'@media (prefers-color-scheme: dark) {{ .rich-adaptive-pre {{ color: #d1d5db; }} }}</style>'
    f'<pre class="rich-adaptive-pre" style="white-space:pre;overflow-x:auto;line-height:normal;{FONT}">{SAMPLE_RICH_HTML}</pre>'
    f'</div>'
)
display(HTML(html))

---
## 6. JS-based detection → inline style override
Detects theme via JS, then directly sets the `<pre>` color. No CSS tricks needed.
Most reliable but requires JS execution.

In [ ]:
DIRECT_JS = (
    "(function(){"
    "var pre=document.currentScript.previousElementSibling;"
    "try{"
    "var b=document.body,t='light';"
    "var jp=b.dataset.jpThemeLight;"
    "if(jp==='false')t='dark';else if(jp==='true')t='light';"
    "if(!t||t==='light'){"
    "var cn=b.className||'';"
    "if(cn.includes('jp-mod-dark'))t='dark';}"
    "var tk=b.getAttribute('data-vscode-theme-kind');"
    "if(tk&&tk.includes('dark'))t='dark';"
    "if(t==='dark')pre.style.color='#d1d5db';"
    "else pre.style.color='#111827';"
    "}catch(e){pre.style.color='inherit';}"
    "})()"
)

pre_style = f'white-space:pre;overflow-x:auto;line-height:normal;{FONT}'
html = (
    f'<div style="margin:12px 0">'
    f'<div style="{FONT}; font-size:0.8em; color:gray; margin-bottom:4px">6. Direct JS detection → set pre color</div>'
    f'<pre style="{pre_style}">{SAMPLE_RICH_HTML}</pre>'
    f'<script>{DIRECT_JS}</script>'
    f'</div>'
)
display(HTML(html))

---
## 7. Actual Rich Progress (live)
Run a real Rich progress bar to see how it looks with the current patch.

In [ ]:
import time
from rich.progress import Progress, BarColumn, TextColumn, MofNCompleteColumn, TimeElapsedColumn, SpinnerColumn

with Progress(
    SpinnerColumn(),
    TextColumn("{task.description}"),
    BarColumn(),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
) as progress:
    task = progress.add_task("Processing", total=20)
    for i in range(20):
        time.sleep(0.05)
        progress.advance(task)

---
## 8. Rich Progress with our patch applied
Same progress bar, but with the format patch applied first.

In [ ]:
from hypergraph.events.rich_progress import _patch_rich_jupyter
_patch_rich_jupyter()

# Show what the patched format looks like
import rich.jupyter as rj
print("Patched format:")
print(rj.JUPYTER_HTML_FORMAT)

In [ ]:
with Progress(
    SpinnerColumn(),
    TextColumn("{task.description}"),
    BarColumn(),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
) as progress:
    task = progress.add_task("Processing (patched)", total=20)
    for i in range(20):
        time.sleep(0.05)
        progress.advance(task)